<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/05_sequences/notebooks/sesion_05_6_integrador_bike_sharing.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/05_sequences/notebooks/sesion_05_6_integrador_bike_sharing.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F05_sequences%2Fnotebooks%2Fsesion_05_6_integrador_bike_sharing.ipynb)

# SI7011 · Sesión 05 · Ejercicio integrador: ¿cuántas horas adelante le gana tu red a la respuesta obvia?
**Juan David Martínez Vargas · Universidad EAFIT**

Un sistema de bicicletas compartidas necesita saber cuántas bicicletas se van a alquilar en la próxima hora, en 6, en 12 y en 24 horas, para redistribuirlas entre estaciones. Tienes dos años de registros por hora de Capital Bikeshare (Washington D. C., 2011–2012) con el clima y el calendario.

La respuesta obvia ya es buena: "lo mismo que a esta hora ayer". La pregunta es **hasta qué horizonte** una red recurrente le gana, y por cuánto.

| Etapa | Qué haces | Artefacto |
|---|---|---|
| Datos | contrato, ficha y huecos (listos) | `data_card.json` |
| Partición | ventanas en el tiempo sin fuga y normalización con el pasado (**TODO 1**) | |
| Modelo | una red recurrente que predice los 4 horizontes a la vez (**TODO 2**), registrada en MLflow | `modelo.pt` |
| Evaluación | MAE por horizonte frente a dos líneas base (**TODO 3**) | tabla en MLflow |
| Despliegue | un pronóstico desde registros crudos, que rechaza los que rompen el contrato (**TODO 4**) | `forecast_raw` |

**Duración:** 60–75 min. **Corre en CPU**: cada entrenamiento tarda menos de un minuto. Cada TODO indica el notebook de la sesión donde está la pieza que necesitas.

In [ ]:
%pip install -q mlflow

In [ ]:
import io, json, os, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
import mlflow

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· MLflow', mlflow.__version__, '· device', device)

## 1. Datos: contrato, ficha y huecos (listo)
`hour.csv` del [Bike Sharing Dataset de UCI](https://archive.ics.uci.edu/dataset/275/) (Fanaee-T y Gama, 2013): una fila por hora, 17 379 filas. La celda lo busca en `data/` (o en `/kaggle/input` si lo agregaste con *Add Input*) y, si no lo encuentra, descarga el zip de UCI.

El **contrato** dice qué debe cumplir cada registro antes de entrar al modelo. Se revisa aquí, sobre los datos de entrenamiento, y otra vez en el despliegue (TODO 4), sobre los registros nuevos.

In [ ]:
ROOTS = [Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]
URL_UCI = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'

def cargar_hour():
    for root in [r for r in ROOTS if r.exists()]:
        hits = sorted(root.rglob('hour.csv'))
        if hits:
            return pd.read_csv(hits[0]), str(hits[0])
    with urllib.request.urlopen(URL_UCI) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    return pd.read_csv(z.open('hour.csv')), URL_UCI

crudo, fuente = cargar_hour()
print('fuente:', fuente, '·', crudo.shape)
crudo.head(3)

In [ ]:
CONTRATO = {
    'columnas': ['dteday', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt'],
    'rangos': {'hr': [0, 23], 'weekday': [0, 6], 'holiday': [0, 1], 'workingday': [0, 1], 'weathersit': [1, 4],
               'temp': [0, 1], 'atemp': [0, 1], 'hum': [0, 1], 'windspeed': [0, 1], 'cnt': [0, 10000]},
    'frecuencia': '1 hora, sin marcas de tiempo repetidas',
}

def con_hora(df):
    """Agrega la marca de tiempo t = fecha + hora y ordena."""
    df = df.copy()
    df['t'] = pd.to_datetime(df['dteday']) + pd.to_timedelta(df['hr'], unit='h')
    return df.sort_values('t').reset_index(drop=True)

def revisar_contrato(df, contrato=CONTRATO):
    """Lanza ValueError si el DataFrame rompe el contrato; devuelve la lista de problemas encontrados (vacía si cumple)."""
    faltan = [c for c in contrato['columnas'] if c not in df.columns]
    if faltan:
        raise ValueError(f'faltan columnas: {faltan}')
    problemas = []
    for c, (lo, hi) in contrato['rangos'].items():
        malos = int(((df[c] < lo) | (df[c] > hi) | df[c].isna()).sum())
        if malos:
            problemas.append(f'{c}: {malos} valores fuera de [{lo}, {hi}]')
    if con_hora(df)['t'].duplicated().any():
        problemas.append('marcas de tiempo repetidas')
    if problemas:
        raise ValueError('; '.join(problemas))
    return problemas

df = con_hora(crudo)
revisar_contrato(df)
horas_esperadas = pd.date_range(df.t.min(), df.t.max(), freq='h')
faltantes = horas_esperadas.difference(df.t)
saltos = (df.t.diff() > pd.Timedelta('1h')).sum()
ficha = {'fuente': fuente, 'filas': len(df), 'desde': str(df.t.min()), 'hasta': str(df.t.max()),
         'horas_faltantes': len(faltantes), 'huecos': int(saltos), 'cnt_media': float(df.cnt.mean())}
json.dump(ficha, open('data_card.json', 'w'), indent=1)
print(json.dumps(ficha, indent=1))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 4.6))
diario = df.set_index('t').cnt.resample('D').sum()
axes[0].plot(diario, lw=0.8); axes[0].set_ylabel('alquileres / día')
semana = df[(df.t >= '2012-06-04') & (df.t < '2012-06-11')]
axes[1].plot(semana.t, semana.cnt, marker='.', ms=3); axes[1].set_ylabel('alquileres / hora')
axes[1].set_title('una semana de junio de 2012', fontsize=9)
plt.tight_layout(); plt.show()

Tres ritmos: el día (picos de las 8 y las 17 los días laborales), la semana y el año. Y una tendencia: 2012 tiene casi el doble de alquileres que 2011. Fíjate en los **huecos**: horas sin registro (fallas del sistema; la más larga, unos días de octubre de 2012 con el huracán Sandy). Ninguna ventana puede cruzar uno.

### Variables de entrada (listo)
En cada hora: la demanda (en escala logarítmica), el clima de esa hora y el calendario. La hora del día y el día de la semana van como seno y coseno, para que las 23 h queden cerca de las 0 h.

Una decisión de contrato importante: la red solo ve el **pasado**. El clima de las próximas 24 horas no está en los datos de entrada, porque al momento del pronóstico no se conoce.

In [ ]:
def variables(df):
    """Matriz (horas × 12) de entrada y la demanda cruda. No normaliza: eso es parte del TODO 1."""
    hr, wd = df['hr'].to_numpy(), df['weekday'].to_numpy()
    X = np.column_stack([
        np.log1p(df['cnt'].to_numpy()),                               # demanda, escala log
        df['temp'], df['atemp'], df['hum'], df['windspeed'],
        (df['weathersit'].to_numpy() - 1) / 3,                         # 1 despejado … 4 tormenta → 0 … 1
        np.sin(2 * np.pi * hr / 24), np.cos(2 * np.pi * hr / 24),
        np.sin(2 * np.pi * wd / 7), np.cos(2 * np.pi * wd / 7),
        df['workingday'], df['holiday'],
    ]).astype(np.float32)
    return X, df['cnt'].to_numpy().astype(np.float32)

NOMBRES = ['log_cnt', 'temp', 'atemp', 'hum', 'windspeed', 'weathersit', 'hr_sin', 'hr_cos', 'wd_sin', 'wd_cos', 'workingday', 'holiday']
X_crudo, cnt = variables(df)
horas = df['t'].to_numpy()
T, HORIZONTES = 72, [1, 6, 12, 24]            # 3 días de entrada; pronóstico a 1, 6, 12 y 24 horas
H_MAX = max(HORIZONTES)
CORTES = {'val': np.datetime64('2012-07-01'), 'test': np.datetime64('2012-10-01')}
I_VAL, I_TEST = int(np.searchsorted(horas, CORTES['val'])), int(np.searchsorted(horas, CORTES['test']))
print(f'{X_crudo.shape[1]} variables · train hasta {CORTES["val"]} ({I_VAL:,} h) · validación hasta {CORTES["test"]} · prueba hasta el final')

## 2. TODO 1 · Partición en el tiempo y ventanas sin fuga · 10 min
Dos piezas, como en los notebooks 1 y 2:

1. `mu, sd`: media y desviación de cada variable, calculadas **solo** con las horas de entrenamiento (`X_crudo[:I_VAL]`).
2. `ventanas(a, b)`: los índices `i` de inicio de ventana tales que la ventana `i … i+T-1` **y** el objetivo más lejano `i+T-1+H_MAX` caen dentro del bloque `[a, b)` **y** no saltan ningún hueco: de `horas[i]` a `horas[i+T-1+H_MAX]` debe haber exactamente `T - 1 + H_MAX` horas.

Pista: el notebook 1 tiene casi lo mismo (sección 3).

In [ ]:
# TODO 1a: estadísticos solo del pasado
mu, sd = None, None

def ventanas(a, b):
    # TODO 1b: inicios de ventana dentro de [a, b) que no cruzan un hueco
    raise NotImplementedError('Completa TODO 1')

In [ ]:
tr, va, te = ventanas(0, I_VAL), ventanas(I_VAL, I_TEST), ventanas(I_TEST, len(horas))
assert mu is not None and np.allclose(mu, X_crudo[:I_VAL].mean(0)), 'mu debe salir solo de las horas de entrenamiento.'
assert tr.max() + T - 1 + H_MAX < I_VAL and va.min() >= I_VAL and te.min() >= I_TEST, 'Una ventana cruza un corte.'
for ini in (tr, va, te):
    assert np.all((horas[ini + T - 1 + H_MAX] - horas[ini]) == np.timedelta64(T - 1 + H_MAX, 'h')), 'Una ventana salta un hueco.'
Xn = (X_crudo - mu) / sd
print(f'ventanas · train {len(tr):,} · validación {len(va):,} · prueba {len(te):,}')

## 3. Las dos líneas base (listo)
- **persistencia:** en $t + H$ habrá lo mismo que ahora, en $t$;
- **misma hora de ayer:** en $t + H$ habrá lo que hubo en $t + H - 24$. Para $H \le 24$ esa hora ya está en la ventana: no hace trampa.

El MAE se mide en **bicicletas por hora**.

In [ ]:
def objetivos(ini):
    """Demanda real en cada horizonte: (n, 4), en bicicletas."""
    return np.stack([cnt[ini + T - 1 + h] for h in HORIZONTES], 1)

def base_persistencia(ini):
    return np.stack([cnt[ini + T - 1]] * len(HORIZONTES), 1)

def base_ayer(ini):
    return np.stack([cnt[ini + T - 1 + h - 24] for h in HORIZONTES], 1)

def mae_h(pred, ini):
    return {h: float(v) for h, v in zip(HORIZONTES, np.abs(pred - objetivos(ini)).mean(0))}

bases_val = {'persistencia': mae_h(base_persistencia(va), va), 'misma hora de ayer': mae_h(base_ayer(va), va)}
pd.DataFrame(bases_val).T.rename(columns=lambda h: f'{h} h').round(1)

**Predice:** ¿en qué horizonte crees que una red le ganará más a la mejor de las dos líneas base? ¿Y en cuál menos?

_Mi predicción:_

## 4. El loop y MLflow (listo)
El modelo recibe una ventana `(B, T, 12)` normalizada y devuelve **4 números**, uno por horizonte: la demanda futura en la misma escala normalizada que la variable `log_cnt` de entrada. `a_bicis` la devuelve a bicicletas.

`entrenar` registra en MLflow los parámetros y el MAE de validación por horizonte en cada época, y guarda `modelo.pt` con todo lo que necesita el despliegue: pesos, `mu`, `sd`, $T$, horizontes, variables y contrato.

In [ ]:
class Ventanas(torch.utils.data.Dataset):
    def __init__(self, inicios):
        self.inicios = inicios
    def __len__(self):
        return len(self.inicios)
    def __getitem__(self, i):
        a = self.inicios[i]
        y = np.array([Xn[a + T - 1 + h, 0] for h in HORIZONTES], dtype=np.float32)   # log_cnt normalizado
        return torch.from_numpy(Xn[a:a + T]), torch.from_numpy(y)

def a_bicis(z):
    """Normalizado → log1p(cnt) → bicicletas."""
    return np.expm1(z * sd[0] + mu[0]).clip(min=0)

@torch.no_grad()
def predecir(modelo, ini):
    modelo.eval()
    salidas = [modelo(x.to(device)).cpu().numpy() for x, _ in torch.utils.data.DataLoader(Ventanas(ini), batch_size=512)]
    return a_bicis(np.concatenate(salidas))

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('S05-bike-sharing')

def entrenar(nombre, construir, params, epocas=12, lr=3e-3):
    torch.manual_seed(0)
    modelo = construir().to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr)
    carga = torch.utils.data.DataLoader(Ventanas(tr), batch_size=128, shuffle=True)
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.log_params({**params, 'T': T, 'horizontes': HORIZONTES, 'epocas': epocas, 'lr': lr,
                           'parametros': sum(p.numel() for p in modelo.parameters())})
        t0 = time.time()
        for ep in range(epocas):
            modelo.train()
            for x, y in carga:
                loss = nn.functional.l1_loss(modelo(x.to(device)), y.to(device))   # MAE en escala normalizada
                opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(modelo.parameters(), 1.0); opt.step()
            m = mae_h(predecir(modelo, va), va)
            mlflow.log_metrics({f'val_mae_{h}h': v for h, v in m.items()}, step=ep)
        print(f'{nombre}: {time.time() - t0:.0f} s · MAE de validación', {f'{h} h': round(v, 1) for h, v in m.items()})
        torch.save({'estado': modelo.state_dict(), 'modelo': params.get('modelo'), 'mu': mu, 'sd': sd, 'T': T,
                    'horizontes': HORIZONTES, 'variables': NOMBRES, 'contrato': CONTRATO}, 'modelo.pt')
        mlflow.log_artifact('modelo.pt')
    return modelo, run.info.run_id

## 5. TODO 2 · Tu modelo recurrente · 15 min
Escribe `Pronosticador`: una red recurrente (`nn.LSTM` o `nn.GRU`, notebook 3) sobre las 12 variables y una capa lineal sobre el último estado $\mathbf h_T$ que devuelva los 4 horizontes. Recibe `(B, T, 12)` y devuelve `(B, 4)`.

Elige el tamaño oculto y el número de capas; anótalo en `params`.

In [ ]:
class Pronosticador(nn.Module):
    def __init__(self, oculta=32, capas=1):
        super().__init__()
        # TODO 2
        raise NotImplementedError('Completa TODO 2')

    def forward(self, x):
        # x: (B, T, 12) → (B, 4)
        raise NotImplementedError('Completa TODO 2')

In [ ]:
x, _ = next(iter(torch.utils.data.DataLoader(Ventanas(tr), batch_size=8)))
assert Pronosticador()(x).shape == (8, len(HORIZONTES)), 'Esperaba una salida (B, 4).'
CONSTRUIR = {'Pronosticador': Pronosticador}
params = {'modelo': 'Pronosticador', 'celda': '...', 'oculta': 32, 'capas': 1}   # describe tu TODO 2
modelo, run_id = entrenar('1-recurrente', CONSTRUIR['Pronosticador'], params)

## 6. TODO 3 · MAE por horizonte frente a las líneas base · 10 min
Escribe `tabla_horizontes(pred, ini)`: recibe las predicciones del modelo en bicicletas `(n, 4)` y los inicios de ventana, y devuelve un `DataFrame` con una fila por horizonte y las columnas `modelo`, `persistencia`, `misma hora de ayer` (MAE en bicicletas) y `mejora` = 1 − MAE del modelo / MAE de la **mejor** línea base en ese horizonte.

Usa `mae_h`, `base_persistencia` y `base_ayer` de la sección 3.

In [ ]:
def tabla_horizontes(pred, ini):
    # TODO 3
    raise NotImplementedError('Completa TODO 3')

In [ ]:
tabla = tabla_horizontes(predecir(modelo, va), va)
assert list(tabla.index) == HORIZONTES and {'modelo', 'persistencia', 'misma hora de ayer', 'mejora'} <= set(tabla.columns)
assert np.isclose(tabla.loc[24, 'misma hora de ayer'], bases_val['misma hora de ayer'][24]), 'Las líneas base no coinciden con la sección 3.'
with mlflow.start_run(run_id=run_id):
    mlflow.log_table(tabla.reset_index(), 'mae_por_horizonte.json')
    mlflow.log_metrics({f'val_mejora_{h}h': float(v) for h, v in tabla['mejora'].items()})
ax = tabla[['modelo', 'persistencia', 'misma hora de ayer']].plot(marker='o', figsize=(6, 3.4))
ax.set_ylabel('MAE de validación (bicicletas / hora)'); plt.tight_layout(); plt.show()
tabla.round(3)

## 7. Cambia una sola cosa · 10 min
Elige **un** cambio, córrelo como `'2-...'` con `entrenar` y compáralo con la tabla de la sección 6. Escribe antes qué esperas. Algunas opciones:

- $T = 24$ en lugar de 72 (vuelve a correr desde la sección 2: las ventanas cambian);
- la otra celda (`nn.LSTM` ↔ `nn.GRU`), o dos capas;
- quita el clima de las entradas (deja solo `log_cnt` y el calendario);
- una pérdida MSE en lugar de MAE.

_Mi cambio y mi predicción:_

In [ ]:
# Tu corrida 2

## 8. Prueba, una sola vez (listo)
El modelo de la sección 5 sobre octubre–diciembre de 2012, que nadie miró.

In [ ]:
tabla_test = tabla_horizontes(predecir(modelo, te), te)
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics({f'test_mae_{h}h': float(v) for h, v in tabla_test['modelo'].items()})
tabla_test.round(3)

## 9. TODO 4 · Despliegue: pronóstico desde registros crudos · 10 min
Escribe `forecast_raw(ckpt, registros)`. Recibe un `DataFrame` con las columnas crudas de `hour.csv` (las del contrato) para las **últimas horas** hasta el momento del pronóstico, y devuelve `{1: bicis, 6: bicis, 12: bicis, 24: bicis}`. Usa solo lo que trae `ckpt`:

1. `revisar_contrato(registros, ckpt['contrato'])`: columnas y rangos;
2. ordenar con `con_hora`, tomar las últimas `ckpt['T']` horas y **rechazar** (`ValueError`) si son menos de `T` o si no son horas consecutivas;
3. `variables(...)`, normalizar con `ckpt['mu']` y `ckpt['sd']`;
4. reconstruir el modelo con `CONSTRUIR[ckpt['modelo']]()`, cargar `ckpt['estado']`, `eval()`, y devolver a bicicletas con `ckpt['mu'][0]` y `ckpt['sd'][0]`.

In [ ]:
ckpt = torch.load(mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path='modelo.pt'), map_location='cpu', weights_only=False)

In [ ]:
@torch.no_grad()
def forecast_raw(ckpt, registros):
    # TODO 4
    raise NotImplementedError('Completa TODO 4')

In [ ]:
# 72 horas crudas que terminan el 2012-11-15 a las 8:00; el real, 1 h después
fin = int(np.searchsorted(horas, np.datetime64('2012-11-15T08:00')))
registros = crudo.iloc[fin - T + 1: fin + 1][CONTRATO['columnas']]
pron = forecast_raw(ckpt, registros)
print({h: round(v) for h, v in pron.items()}, '· real a 1 h:', int(cnt[fin + 1]))
assert set(pron) == set(HORIZONTES) and all(v >= 0 for v in pron.values())
for malo, motivo in [(registros.drop(columns=['hum']), 'una columna menos'),
                     (crudo.iloc[fin - T - 9: fin + 1][CONTRATO['columnas']].drop(index=fin - 20), 'un hueco'),
                     (registros.assign(temp=registros.temp * 40), 'temperatura en °C en lugar de normalizada')]:
    try:
        forecast_raw(ckpt, malo)
        raise AssertionError(f'forecast_raw aceptó registros con {motivo}')
    except ValueError as e:
        print(f'rechazado ({motivo}):', e)

## 10. Preguntas
1. ¿Se cumplió tu predicción de la sección 3? ¿En qué horizonte ganó más la red y en cuál menos? ¿Por qué la mejora se reduce a 24 h, donde "misma hora de ayer" ya es la mejor línea base?
2. A 1 h la persistencia es muy buena y a 24 h lo es "misma hora de ayer". ¿Qué parte de la ventana de 72 h crees que usa la red en cada caso? (Piensa en la Parte 2 y en la F11.)
3. El contrato no deja entrar el clima **futuro**. Si un servicio meteorológico te diera el pronóstico del clima para las próximas 24 h, ¿cómo lo incluirías sin hacer trampa en la evaluación?
4. Los datos de prueba (octubre–diciembre de 2012) tienen más demanda que todo lo que el modelo vio en 2011. ¿Cómo lo afecta? ¿Qué parte del preprocesamiento lo absorbe en parte?
5. `forecast_raw` rechaza registros con huecos. En producción el sistema a veces pierde una hora. ¿Rechazarías, rellenarías el hueco o pronosticarías con la ventana más corta? ¿Qué riesgo tiene cada opción?

_Respuestas:_

**Si quieres ir más allá:** agrega una línea base más fuerte, "misma hora hace una semana" (necesitas  \ge 168$), y mira si la red le sigue ganando a 24 h.